# Citi Velocity timeseries straight from the Excel add-in

Every number in this notebook comes off the wire on the spot. `direct="live"`
bypasses **every** cache tier - the computed-timeseries store, its DuckDB mirror,
Supabase L2, and the Velocity tag parquet - and writes nothing back. If the add-in
cannot serve, it **raises**; it never degrades to a cached number wearing a live
face.

What this notebook shows, in order:

| | |
|---|---|
| **1** | connect, or fall back to the packaged fake |
| **2** | `CitiVeloTags` - all 67,425 catalog tags as enums you can tab through |
| **3** | those members inside `CitiVeloQuery`: outright, curve, fly, cross-curve spread |
| **4** | **EOD** direct, through `TimeseriesBuilder` |
| **5** | the proof that nothing was read from a cache and nothing was written to one |
| **6** | **intraday** MI01 direct - and why it must *not* go through `TimeseriesBuilder` |
| **7** | the **live** append: the newest print, stitched onto the history |
| **8** | every way `direct=` refuses |

## Before you run this

This drives **your own Excel process** over COM. There is no headless path: the
Velocity login is gated on the ribbon loading, and spawned Excel instances never
register the `CV*` UDFs.

* Excel must be open and **signed in to Velocity**.
* After an Excel restart the login takes **~13 minutes** and logs nothing in
  between. If `connect()` fails right after a restart, wait - do not retry in a
  loop.
* Never interrupt a cell mid-call. That wedges Excel's OLE server for ~15 minutes.
* Every Excel access in this process serialises behind one lock. If a warm or a
  backfill is running, let it finish.

If no signed-in add-in is reachable the next cell falls back to the packaged
**fake**, so the whole notebook still runs - and says so on every figure.

In [ ]:
%load_ext autoreload
%autoreload 2

import datetime
import warnings

import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
import numpy as np
import pandas as pd
import pytz

plt.style.use("ggplot")
pylab.rcParams.update({
    "legend.fontsize": "medium", "figure.figsize": (16, 4.5),
    "axes.labelsize": "medium", "axes.titlesize": "medium",
    "xtick.labelsize": "medium", "ytick.labelsize": "medium",
})
pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 40)
warnings.filterwarnings("ignore", category=UserWarning)

NYC = pytz.timezone("America/New_York")

import sys
sys.path.append("../../")

In [ ]:
from MDP.CitiVelocityExcel import tags as T
from MDP.CitiVelocityExcel.com_client import CitiVelocityExcelClient
from MDP.CitiVelocityExcel.errors import CitiVelocityError
from MDP.CitiVelocityExcel.mdp import CitiVelocityMDP
from MDP.CitiVelocityExcel.quotes import CitiVeloQuotes
from Query.CitiVelocity import (
    CitiVeloQuery,
    CitiVeloStructure,
    CitiVeloTags,
    CitiVeloValue,
)
from TB.CitiVelocityTB import CitiVelocityTB
from TB.direct_mode import DirectModeError
from TB.TimeseriesBuilder import TimeseriesBuilder

## 1. Connect, or fall back to the fake

`CitiVelocityMDP()` is the ordinary **cache-backed** provider. You hand that one
to `TimeseriesBuilder`; it derives the cache-less sibling itself, per call, and
throws it away afterwards. Passing `.direct_mdp()` yourself also works - it is
reused as-is rather than siblinged twice - and that is what sections 6 and 7 do,
because they go under the builder.

In [ ]:
import os

# Set ARBS_CITIVELO_FORCE_FAKE=1 to take the synthetic path on a machine that HAS a
# signed-in add-in. Useful for re-running the notebook without spending Excel's
# budget, and it is how this notebook is checked end to end.
FORCE_FAKE = os.environ.get("ARBS_CITIVELO_FORCE_FAKE", "").strip().lower() in {"1", "true", "yes"}

LIVE = False
if FORCE_FAKE:
    print("ARBS_CITIVELO_FORCE_FAKE is set - not connecting to Excel")
else:
    try:
        client = CitiVelocityExcelClient.connect(attempts=1, readiness_timeout=60.0)
        LIVE = True
        print("connected to a signed-in Citi Velocity add-in")
    except Exception as exc:
        print(f"NO LIVE ADD-IN ({type(exc).__name__}: {exc})")

if not LIVE:
    print(">>> falling back to the packaged fake - every number below is SYNTHETIC <<<")

BANNER = "" if LIVE else "   [SYNTHETIC]"

CURVE = "USD_SOFR"
PAR_GRID = T.ois_par_grid(CURVE)                  # 44 tenors, one CVTSHIST call
TEN_YEAR = str(CitiVeloTags.OIS.USD_SOFR.PAR_10Y)  # the enum and the string agree
UST_10Y = T.tsy_otr("10Y")                        # the intraday subject

if not LIVE:
    from MDP.CitiVelocityExcel.catalog import tenor_years
    from MDP.CitiVelocityExcel.testing import FakeExcelApp, FakeVelocityData

    _days = pd.bdate_range("2025-08-01", "2026-08-20")
    _series = {}
    # The SOFR par grid, plus the Fed Funds 10y the cross-curve spread needs. A tag
    # the fake does not serve makes the direct request RAISE rather than quietly
    # dropping a leg - which is the contract, and is how this list was found.
    for _tag in list(PAR_GRID) + [str(CitiVeloTags.OIS.USD_FEDFUND.PAR_10Y)]:
        _y = tenor_years(_tag.rsplit(".", 1)[-1])
        _level = 3.60 + 0.90 * (1.0 - np.exp(-_y / 3.0))
        _level -= 0.04 if "FEDFUND" in _tag else 0.0
        # A per-tenor phase, so curves and flies MOVE. A single shared shock would
        # make every structure a flat line and the plots would prove nothing.
        _phase = 0.35 * np.log1p(_y) + (0.7 if "FEDFUND" in _tag else 0.0)
        _series[_tag] = pd.Series(
            _level + 0.05 * np.sin(np.arange(len(_days)) / 40.0 + _phase)
            + 0.01 * np.sin(np.arange(len(_days)) / 11.0 + 2 * _phase),
            index=_days,
        )
    # A separate MINUTE series for the intraday subject. Two days is inside the
    # measured MI01 span cliff (6 days), which the fake reproduces.
    _mins = pd.date_range("2026-08-19 08:00", "2026-08-20 16:00", freq="1min")
    _series[UST_10Y] = pd.Series(
        4.25 + 0.02 * np.sin(np.arange(len(_mins)) / 90.0)
        + 0.001 * np.cos(np.arange(len(_mins)) / 7.0),
        index=_mins,
    )
    client = CitiVelocityExcelClient(app=FakeExcelApp(FakeVelocityData(series=_series)))

# The cache-backed provider. Note what it is NOT: nothing below reads from its
# cache, and section 5 proves the cache root is untouched afterwards.
mdp = CitiVelocityMDP(quotes=CitiVeloQuotes(client=client))
print(f"tag cache root : {mdp.quotes.cache.base_dir}")
print(f"mdp.direct     : {mdp.direct}   (False - this is the ordinary provider)")

## 2. `CitiVeloTags` - every catalog tag, as an enum

67,425 tags across 33 `RATES.*` families, generated from the committed catalog by
`scripts/gen_citivelo_tag_enums.py`. They are `StrEnum` members, so a member **is**
its tag string - it goes anywhere a string goes.

Nothing is imported until you touch it: the package costs a few milliseconds, and
each family is loaded on first attribute access.

**Type the dot and let the language server drive.** A node holding more than 1,024
tags splits on its next segment, so the depth varies (`TSY` is flat at the family
level; `VOL.USD.OTM_RFR.NORMALABSOLUTE` is four levels down). Namespaces and leaf
enums answer the same five questions - `node()`, `children()`, `tags()`, `search()`,
`find()` - so walking never has to know which one it is holding.

In [ ]:
print(f"{CitiVeloTags.TOTAL:,} tags | {len(CitiVeloTags.FAMILIES)} families "
      f"| catalog fingerprint {CitiVeloTags.FINGERPRINT[:12]}")
print()
print("biggest families:", ", ".join(f"{f} {n:,}" for f, n in CitiVeloTags.counts()[:6]))
print()
print("RATES.OIS branches   :", ", ".join(CitiVeloTags.OIS.children()[:8]), "...")
print("RATES.OIS.USD_SOFR   :", ", ".join(CitiVeloTags.OIS.USD_SOFR.children()[:8]), "...")
print()
print("one member  :", repr(CitiVeloTags.OIS.USD_SOFR.PAR_10Y))
print("str()       :", str(CitiVeloTags.OIS.USD_SOFR.PAR_10Y))
print("is a str    :", isinstance(CitiVeloTags.OIS.USD_SOFR.PAR_10Y, str))

In [ ]:
m = CitiVeloTags.OIS.USD_SOFR.PAR_10Y
print(f"family {m.family} | segments {m.segments} | tenor {m.tenor}")
print(f"kind {m.kind().name} | repriceable {m.is_repriceable} | verification {m.verification()!r}")
print()

# Search inside a family. Scope it: unscoped search loads all 33.
from MDP.CitiVelocityExcel.catalog import sort_tenors
hits = CitiVeloTags.search("*USD_SOFR.SWAP_SPREAD*", family="OIS", limit=20)
print("swap-spread tenors:", sort_tenors([h.tenor for h in hits]))
print()

# The inverse of the name mangling: a tag out of a log becomes a typed member,
# loading only its own family.
print("find() ->", repr(CitiVeloTags.find("RATES.VOL.USD.ATM_RFR.BLACK.1Y.10Y")))
print()

# The legacy VOL branches are kept because they are real recorded structure, and
# flagged because they serve NO DATA - reaching for ATM when you meant ATM_RFR
# gives an empty series rather than an error.
for tag in ("RATES.VOL.USD.ATM.BLACK.1Y.10Y", "RATES.VOL.USD.ATM_RFR.BLACK.1Y.10Y"):
    print(f"  deprecated={str(CitiVeloTags.find(tag).deprecated):<5}  {tag}")

In [ ]:
# A miss never just says "no". It names what is actually there.
for bad in ("RATES.OIS.USD_SOFR.PAR.99Y", "RATES.VOL.USD.NOT_A_BRANCH.BLACK.1Y.10Y"):
    try:
        CitiVeloTags.find(bad)
    except KeyError as exc:
        print(f"{bad}\n    {exc}\n")

# The enum is a discovery surface, NOT a gate. Two ways a real tag can sit outside
# it, both accepted everywhere a member is:
newer = "RATES.OIS.USD_SOFR.SOMETHING_CITI_ADDED_TODAY.10Y"
print("newer than the catalog :", CitiVeloQuery(tag=newer).tag)

# ...and the Function Builder walk was depth-capped per family, so a tags.py builder
# can legitimately go one segment deeper than the walk recorded. TSY is the case:
print("catalog recorded       :", CitiVeloTags.TSY.TSY_OTR_10Y.value)
print("what tags.tsy_otr gives:", T.tsy_otr("10Y"), " <- one segment deeper")
print("in the enum            :", T.tsy_otr("10Y") in set(CitiVeloTags.TSY.tags()))

## 3. The members inside `CitiVeloQuery`

`tag=` takes a member or a raw string and normalises to a plain `str`, so nothing
downstream can tell which spelling was used. `CitiVeloQuery.Tags` is the same tree,
reachable from the class you already imported.

A `SPREAD` is where the enum earns its keep: a cross-curve structure is exactly
what you build by picking two members.

> **One trap the enum makes easier to reach.** About half of these tags classify as
> `RAW` - the pricer has no model for them, so they carry a unitless `RATIO` and the
> structure layer's same-unit guard cannot see inside. A `SPREAD` across two
> *unrelated* `RAW` families nets and returns a number. Check `.is_repriceable`
> and `.kind()` before you build one.

In [ ]:
queries = [
    CitiVeloQuery(tag=CitiVeloQuery.Tags.OIS.USD_SOFR.PAR_10Y, name="10y"),
    CitiVeloQuery(
        citi_index=CURVE, structure=CitiVeloStructure.CURVE,
        structure_kwargs={"front_tenor": "2Y", "back_tenor": "10Y"}, name="2s10s",
    ),
    CitiVeloQuery(
        citi_index=CURVE, structure=CitiVeloStructure.FLY,
        structure_kwargs={"front_tenor": "2Y", "belly_tenor": "5Y", "back_tenor": "10Y"},
        name="2s5s10s",
    ),
    CitiVeloQuery(
        structure=CitiVeloStructure.SPREAD,
        structure_kwargs={"legs": [
            {"tag": CitiVeloTags.OIS.USD_SOFR.PAR_10Y},
            {"tag": CitiVeloTags.OIS.USD_FEDFUND.PAR_10Y},
        ]},
        name="SOFR-FF 10y",
    ),
]

q = queries[0]
print("tag stored as :", type(q.tag).__name__, repr(q.tag))
print("same as string:", q == CitiVeloQuery(tag='RATES.OIS.USD_SOFR.PAR.10Y', name="10y"))
print()

# The router resolves every leg from the catalog alone - no market needed - so you
# can see which queries take the one-shot tag read before fetching anything.
plan = CitiVelocityTB(mdp, show_tqdm=False).plan(queries)
print(plan.summary().to_string(index=False))
print(f"\n{len(plan.tags)} distinct tags across {len(queries)} queries")

## 4. EOD, straight from the add-in

Pass the **plain** `CitiVelocityMDP` under the key `"CITIVELO"` and add
`direct="live"`. The builder derives the cache-less sibling for you.

Two things move under `direct=`, both on purpose:

* the frame is a **full grid** - one row per reference point, one column per query -
  with an explicit `NaN` wherever nothing could be served, instead of the shorter
  frame the cached path returns;
* a column that is `NaN` at *every* reference point **raises**. An empty direct
  result is not a pass.

`n_jobs` is accepted and ignored here, and `price_point` on the query is **not**
forwarded on this path - the wire always gets `CLOSE`. For OHLC, go through
`quotes.frame(..., price_point="OPEN")` as section 6 does.

In [ ]:
END = datetime.date(2026, 8, 20)
START = END - datetime.timedelta(days=180)

calls_before = client.calls
eod = TimeseriesBuilder().get_timeseries(
    START, END, queries,
    mdps={"CITIVELO": mdp},     # the key must be the literal "CITIVELO"
    direct="live",
)
print(f"\nCV* wire calls for the whole request: {client.calls - calls_before}{BANNER}")
print(f"frame: {eod.shape[0]} rows x {eod.shape[1]} cols, index {type(eod.index).__name__}")
eod.tail(4)

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 7), sharex=True)
eod["10y"].plot(ax=ax1, lw=1.2, color="tab:blue")
ax1.set_ylabel("%")
ax1.set_title(f"USD SOFR 10y par, read direct from the add-in{BANNER}")
for col, colour in (("2s10s", "tab:green"), ("2s5s10s", "tab:red")):
    eod[col].plot(ax=ax2, lw=1.2, color=colour, label=col)
ax2.set_ylabel("bp")
ax2.axhline(0, lw=0.6, ls="--", color="grey")
ax2.legend()
ax2.set_title("structures - multi-leg packages report in basis points")
plt.tight_layout()

## 5. Nothing was read from a cache, and nothing was written to one

"Fresh data came back" proves nothing - a refetch-and-merge also calls the
transport and also returns fresh values. What distinguishes a bypass is that the
store is **untouched in both directions**.

So: snapshot the tag-cache tree (relative path -> size, mtime) before and after,
and check the reader underneath really had no cache to use.

In [ ]:
import os

def snapshot(root):
    root = str(root)
    out = {}
    for base, _dirs, files in os.walk(root):
        for name in files:
            p = os.path.join(base, name)
            st = os.stat(p)
            out[os.path.relpath(p, root)] = (st.st_size, st.st_mtime_ns)
    return out

root = mdp.quotes.cache.base_dir
before = snapshot(root)

_ = TimeseriesBuilder().get_timeseries(
    END - datetime.timedelta(days=10), END, [queries[0]],
    mdps={"CITIVELO": mdp}, direct="live",
)
after = snapshot(root)

print(f"cache root      : {root}")
print(f"files before/after: {len(before)} / {len(after)}")
print(f"IDENTICAL        : {before == after}")
if before != after:
    changed = {k for k in set(before) | set(after) if before.get(k) != after.get(k)}
    print("  changed:", sorted(changed)[:5])
print()

# The reader the direct request actually used.
direct_reader = mdp.direct_mdp().quotes
print(f"is_direct        : {direct_reader.is_direct}   (built to bypass, not merely cache-less)")
print(f"cache            : {direct_reader.cache}       (None - nothing to read, nothing to write)")
print(f"parent unchanged : mdp.direct is still {mdp.direct}")

In [ ]:
# And this is not academic. On 2026-08-21 this very cache was found serving swaption
# NORMAL VOL under the OIS par-rate tags: 44 `RATES.OIS.USD_SOFR.PAR.*` parquets, 2,613
# poisoned days interleaved with good ones, so the series still plotted plausibly. A
# cached read returned ~76 where the true 30y rate was ~4.53.
#
# Direct mode reads through that correctly because it has no cache to be wrong. The
# sanity rule is cheap and worth keeping wherever these tags are read:
rates = eod["10y"].dropna()
assert not rates.empty, "no 10y values came back at all"
outside = rates[(rates < 0.0) | (rates > 15.0)]
print(f"10y par rate range: {rates.min():.4f} .. {rates.max():.4f} %{BANNER}")
print(f"values outside 0-15%: {len(outside)}   <- a USD SOFR par rate outside that is not a rate")
assert outside.empty, f"{len(outside)} value(s) are not par rates: {outside.head().tolist()}"

## 6. Intraday MI01 - and why it must *not* go through `TimeseriesBuilder`

**This is the trap.** `BaseTimeseriesTB` truncates every reference point to a
`datetime.date` before assembling the frame, then groups by it. Hand it 1,500
minute reference points and you get back a frame with **one row per day** - and the
progress bar ticks 1,500 times on the way, so it looks like it worked.

There is no fix at the query level; it is the wrong layer. Go under it: the direct
provider's reader returns a real `DatetimeIndex` at whatever granularity you asked
for.

`MI01` is the finest granularity for every family. `SE10` describes the *streaming*
feed and `CVTSHIST` rejects it outright. Note also the measured **span cliff**: an
`MI01` request spanning more than 6 days is silently served at 10-minute spacing.

In [ ]:
live = mdp.direct_mdp()          # cache-less sibling; BORROWS the COM client

DAY = datetime.date(2026, 8, 20)
open_, close_ = (datetime.datetime.combine(DAY, datetime.time(h, 0)) for h in (8, 16))

# --- the trap, demonstrated -----------------------------------------------
minute_points = pd.date_range(open_, close_, freq="1min").to_pydatetime().tolist()
trap = CitiVelocityTB(live, show_tqdm=False, direct="live").get_timeseries(
    open_, close_,
    [CitiVeloQuery(tag=UST_10Y, freq="MI01", name="UST 10y")],
    timestamps=minute_points,
)
print(f"asked for {len(minute_points):,} minute reference points -> got {len(trap)} row(s)")
print(f"index type: {type(trap.index).__name__}   <- date, not datetime\n")

# --- the correct path ------------------------------------------------------
intraday = live.quotes.frame([UST_10Y], "MI01", start=open_, end=close_)

assert isinstance(intraday.index, pd.DatetimeIndex), "not a minute index"
assert intraday.groupby(intraday.index.date).size().max() > 1, "collapsed to EOD"
gaps = pd.Series(intraday.index).diff().dropna()
print(f"{len(intraday):,} rows, finest observed gap {gaps.min()}{BANNER}")
intraday.tail(3)

In [ ]:
ax = intraday.plot(figsize=(16, 4), lw=0.8, legend=False, color="tab:purple")
ax.set_title(f"UST 10y on-the-run, 1-minute, direct from the add-in - {DAY:%Y-%m-%d}{BANNER}")
ax.set_ylabel("yield, %")
plt.tight_layout()

## 7. The live append - the newest print, stitched onto the history

`direct=` deliberately **refuses** `end="live"`: that is an IRS-only splice that
stitches a live tail onto a *cached* history, which is exactly the mixture direct
mode exists to rule out.

The honest equivalent is to re-read the tail. Ask for the last couple of days at
`MI01` and take the final row; the newest `MI01` row has been measured at about a
minute old. Merge it **incoming-wins**, matching the tag cache's own semantics, so
re-requesting an overlapping span *corrects* a partially-published row rather than
duplicating it.

In [ ]:
tail = live.quotes.frame([UST_10Y], "MI01", period="2D")   # "2D" is inside the 6-day cliff
last_stamp = tail.index[-1]
last_value = float(tail[UST_10Y].iloc[-1])

history = intraday.copy()
was = len(history)
history = history.reindex(history.index.union([last_stamp]))
history.loc[last_stamp, UST_10Y] = last_value           # incoming wins on a collision

age = pd.Timestamp.now() - last_stamp
print(f"newest print : {last_value:.5f} at {last_stamp}  (age {age}){BANNER}")
print(f"history      : {was} -> {len(history)} rows "
      f"({'appended' if len(history) > was else 'corrected in place'})")
history.tail(3)

In [ ]:
# CVLATEST is the alternative, and it is worth knowing what it costs.
#   * it publishes NO timestamp - you self-stamp, so you cannot tell a curve that
#     stopped ticking four hours ago from one that ticked a second ago;
#   * it returns 5 decimal places against CVTSHIST's full double, so appending one
#     onto a CVTSHIST history mixes precisions;
#   * a dead tag comes back (None, None) for itself only.
if LIVE:
    value, stamp = live.quotes.latest([UST_10Y])[UST_10Y]
    print(f"CVLATEST : {value}   stamp from the add-in: {stamp}  <- None, always")
    print(f"           self-stamped {datetime.datetime.now()}")
    print(f"MI01 tail: {last_value}   (full precision, and its own real timestamp)")
else:
    print("skipped: CVLATEST needs the live add-in" + BANNER)

## 8. Every way `direct=` refuses

The vocabulary is closed and case-insensitive, in the style of `Caching/l2_policy`.
A typo in an opt-**in** must not silently resolve to "serve from cache", because
you would then believe a cached number was live - so anything outside the
vocabulary raises rather than defaulting to the safe-looking side.

Every refusal below fires **before** any transport or cache is touched.

In [ ]:
def refuses(label, fn):
    try:
        fn()
    except Exception as exc:
        first = str(exc).split(".")[0]
        print(f"{label:<34} {type(exc).__name__}: {first[:96]}")
    else:
        print(f"{label:<34} *** DID NOT RAISE ***")

tb = TimeseriesBuilder()
one = [queries[0]]

refuses("direct='of'  (typo)",
        lambda: tb.get_timeseries(START, END, one, mdps={"CITIVELO": mdp}, direct="of"))
refuses("direct='no-cache'  (plausible)",
        lambda: tb.get_timeseries(START, END, one, mdps={"CITIVELO": mdp}, direct="no-cache"))
refuses("direct + ignore_cache",
        lambda: tb.get_timeseries(START, END, one, mdps={"CITIVELO": mdp},
                                  direct="live", ignore_cache=True))
refuses("direct + end='live'",
        lambda: tb.get_timeseries(START, "live", one, mdps={"CITIVELO": mdp}, direct="live"))
refuses("wrong mdps key ('citivelo')",
        lambda: tb.get_timeseries(START, END, one, mdps={"citivelo": mdp}, direct="live"))
refuses("cache-backed mdp under direct",
        lambda: CitiVelocityTB(mdp, show_tqdm=False).get_timeseries(
            START, END, one, direct="live"))
refuses("a foreign enum as tag=",
        lambda: CitiVeloQuery(tag=CitiVeloValue.QUOTE))

print()
# And the tokens that DO mean off, so an explicit opt-out is never a typo either.
print("off:", "'' off 0 false f no n none null disabled cached cache  + None/False")
print("on :", "1 on true t yes y live direct addin add_in add-in bypass  + True")

In [ ]:
# A non-CITIVELO query is refused rather than quietly served from ITS cache. FRB and
# IRS read through their own multi-tier stores and have no equivalent bypass, and a
# partial guarantee would be worse than none.
from Query.IRSwaps.IRSwapQuery import IRSwapQuery
from Query.IRSwaps.IRSwapValue import IRSwapValue

refuses("a mixed CITIVELO + IRS basket",
        lambda: tb.get_timeseries(
            START, END,
            [queries[0], IRSwapQuery(curve="USD-SOFR-1D", tenor="10Y", value=IRSwapValue.RATE)],
            mdps={"CITIVELO": mdp}, direct="live"))

## 9. Tidy up

Closes the scratch workbook after a drain pause. Nothing is ever cleared or
deleted: tearing a region down while the add-in's queued `ExcessClr`/`Format`/
`AutoFit` actions are outstanding is itself an `AccessViolation` trigger, and it
takes the whole Excel process with it.

The direct sibling borrowed the client rather than opening a second session, so
there is only one thing to close.

In [ ]:
live.close()      # borrowed client - a no-op for the session
mdp.close()
print("scratch workbook closed; your own workbooks were never touched")